# Multi-Security TLH Prototype
### Vise Capital // UTexas MSBA

---

**Purpose:** Demonstrate the core mechanics of tax-loss harvesting (TLH) across a 10-security portfolio in the clearest possible way.

This is an intentionally simple prototype — not a production system. The goal is to make the logic easy to read, trace, and extend.

**What this notebook does:**
1. Holds 10 securities simultaneously, each as an independent position
2. Checks each position for TLH opportunities at every calendar rebalance event
3. When a position's unrealized loss exceeds the threshold, switches it into its mapped proxy
4. Enforces IRS wash sale rules across the **entire portfolio** — a sold symbol is blocked portfolio-wide
5. Logs every rebalance check for every position — nothing is silent

**Key design decisions:**
- Each position is tracked independently (own `current_symbol`, `shares`, `cost_basis`)
- The wash sale log is **shared** across all positions (IRS rule applies to the taxpayer, not to individual lots)
- No weight rebalancing between positions — each lot is managed on its own

---

**Notebook structure:**
1. Imports
2. Config
3. Load proxy lookup table
4. Load price data
5. Helper functions
6. Main simulation function
7. Run the prototype
8. Event log
9. Portfolio history
10. Notes for future extension

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

print("Libraries loaded.")

## 2. Config

All parameters live here. Change these to run different scenarios.

**`PORTFOLIO`** is a dict of `{starting_symbol: weight}`. Weights are normalized automatically so they don't need to sum to exactly 1.

In [ ]:
# ─── File Paths ──────────────────────────────────────────────────────────────
PRICE_DATA_PATH   = "../../../data/price_data.parquet"
PROXY_LOOKUP_PATH = "../../../data/proxy_lookup.xlsx - proxy_lookup.csv"

# ─── Portfolio ───────────────────────────────────────────────────────────────
# 10 securities across equity, fixed income, and alternatives.
# Weights are relative — they get normalized to sum to 1.0.
PORTFOLIO = {
    "SPY": 0.15,   # US Large Cap Equity
    "QQQ": 0.15,   # US Large Cap Tech
    "EFA": 0.10,   # International Developed Equity
    "VWO": 0.10,   # Emerging Markets Equity
    "AGG": 0.10,   # US Aggregate Bond
    "LQD": 0.10,   # Investment Grade Corporate Bond
    "HYG": 0.10,   # High Yield Bond
    "IEF": 0.10,   # US Intermediate Treasury
    "TLT": 0.05,   # US Long Treasury
    "GLD": 0.05,   # Gold
}

# ─── Simulation Parameters ───────────────────────────────────────────────────
START_DATE        = "2020-01-01"
END_DATE          = "2025-12-31"

REBALANCE_FREQ    = "Monthly"     # "Daily" | "Weekly" | "Monthly" | "Quarterly"

TLH_THRESHOLD     = 0.03          # Trigger TLH if unrealized loss exceeds this
                                   # 0.03 = 3% loss triggers a harvest

WASH_SALE_DAYS    = 30            # IRS wash sale window in days

INITIAL_VALUE     = 1_000_000.0   # Total starting portfolio value in dollars

# ─── Proxy Lookup Config ─────────────────────────────────────────────────────
PROXY_SNAPSHOT_DATE = "12/24/2025"  # Snapshot of the proxy table to use
PRICE_FIELD         = "PRICECLOSE"

# ─── Normalize weights and compute position sizes ────────────────────────────
total_weight = sum(PORTFOLIO.values())
PORTFOLIO = {sym: w / total_weight for sym, w in PORTFOLIO.items()}

print("Portfolio configuration:")
print(f"  {'Symbol':<8}  {'Weight':>8}  {'Initial Value':>14}")
print(f"  {'─'*36}")
for sym, w in PORTFOLIO.items():
    print(f"  {sym:<8}  {w:>8.1%}  ${INITIAL_VALUE * w:>13,.0f}")
print(f"  {'─'*36}")
print(f"  {'TOTAL':<8}  {'100.0%':>8}  ${INITIAL_VALUE:>13,.0f}")
print()
print(f"  Period         : {START_DATE} to {END_DATE}")
print(f"  Rebalance freq : {REBALANCE_FREQ}")
print(f"  TLH threshold  : {TLH_THRESHOLD:.0%} unrealized loss")
print(f"  Wash sale      : {WASH_SALE_DAYS} days")
print(f"  Proxy snapshot : {PROXY_SNAPSHOT_DATE}")

## 3. Load Proxy Lookup Table

The proxy lookup maps each security to its TLH substitute(s). We use `order=1` as the primary proxy.

In [ ]:
proxy_raw = pd.read_csv(PROXY_LOOKUP_PATH)

proxy_df = (
    proxy_raw[
        (proxy_raw["as_of_date"] == PROXY_SNAPSHOT_DATE)
        & (proxy_raw["lookup_type"] == "TLH_SUB")
        & (proxy_raw["order"] == 1)
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Proxy table loaded: {len(proxy_df)} primary TLH substitutes (snapshot: {PROXY_SNAPSHOT_DATE})")
print()
print("Proxies for the configured portfolio:")
print(f"  {'Symbol':<8}  {'Primary Proxy':<14}  {'Proxy\'s Proxy':<14}")
print(f"  {'─'*40}")
for sym in PORTFOLIO:
    row = proxy_df[proxy_df["symbol"] == sym]
    if row.empty:
        print(f"  {sym:<8}  {'NO PROXY':14}")
        continue
    px = row.iloc[0]["lookup_symbol"]
    # Also show what the proxy maps back to (the chain one level deeper)
    px_row = proxy_df[proxy_df["symbol"] == px]
    px2 = px_row.iloc[0]["lookup_symbol"] if not px_row.empty else "—"
    print(f"  {sym:<8}  {px:<14}  {px2:<14}")

## 4. Load Price Data

We load `price_data.parquet`, filter to active securities, and build a wide price table (dates × tickers).
All returns are derived from this local file — no external data fetching.

In [ ]:
_COLS = ["TICKERSYMBOL", "PRICEDATE", PRICE_FIELD, "TRADINGITEMSTATUSID"]
raw = pd.read_parquet(PRICE_DATA_PATH, columns=_COLS)

raw["PRICEDATE"]     = pd.to_datetime(raw["PRICEDATE"])
raw[PRICE_FIELD]     = pd.to_numeric(raw[PRICE_FIELD], errors="coerce")
raw["TICKERSYMBOL"]  = raw["TICKERSYMBOL"].str.strip().str.upper()
raw = raw[raw["TRADINGITEMSTATUSID"].isin([1, 15])].copy()
raw = raw.dropna(subset=[PRICE_FIELD])

date_mask = (
    (raw["PRICEDATE"] >= pd.Timestamp(START_DATE))
    & (raw["PRICEDATE"] <= pd.Timestamp(END_DATE))
)
raw = raw[date_mask]

prices_wide = (
    raw.pivot_table(
        index="PRICEDATE",
        columns="TICKERSYMBOL",
        values=PRICE_FIELD,
        aggfunc="last",
    )
    .sort_index()
    .ffill()
)

print("Price data loaded:")
print(f"  Trading days : {len(prices_wide)}")
print(f"  Symbols      : {prices_wide.shape[1]}")
print(f"  Date range   : {prices_wide.index[0].date()} → {prices_wide.index[-1].date()}")
print()

# Validate every portfolio security and its proxy are present in the price data
print("Data availability check:")
all_ok = True
for sym in PORTFOLIO:
    sym_ok  = sym in prices_wide.columns
    row     = proxy_df[proxy_df["symbol"] == sym]
    px      = row.iloc[0]["lookup_symbol"] if not row.empty else None
    px_ok   = px in prices_wide.columns if px else False
    status  = "✓" if (sym_ok and px_ok) else "⚠"
    print(f"  {status}  {sym:<8} (ok={sym_ok})  proxy={str(px):<8} (ok={px_ok})")
    if not (sym_ok and px_ok):
        all_ok = False

print()
print("All symbols and proxies confirmed." if all_ok else "WARNING: some symbols or proxies missing from price data.")

## 5. Helper Functions

Same four helpers as before — unchanged from the single-security version.
The multi-security logic lives entirely in the simulation function.

In [ ]:
def get_proxy(symbol: str, proxy_df: pd.DataFrame):
    """
    Return the primary TLH substitute (order=1) for a given symbol.
    Returns the proxy ticker string, or None if no proxy is defined.
    """
    row = proxy_df[proxy_df["symbol"] == symbol]
    if row.empty:
        return None
    return row.iloc[0]["lookup_symbol"]


def generate_rebalance_dates(trading_dates: pd.DatetimeIndex, freq: str) -> list:
    """
    Return the subset of trading dates on which a rebalance check occurs.
    Triggers on the first trading day of each new period.
    Supported frequencies: "Daily", "Weekly", "Monthly", "Quarterly"
    """
    dates = pd.DatetimeIndex(trading_dates)
    if len(dates) < 2:
        return []

    rebal = []

    if freq == "Daily":
        return list(dates[1:])

    elif freq == "Weekly":
        prev = (dates[0].year, dates[0].isocalendar()[1])
        for dt in dates[1:]:
            cur = (dt.year, dt.isocalendar()[1])
            if cur != prev:
                rebal.append(dt)
            prev = cur

    elif freq == "Monthly":
        prev = (dates[0].year, dates[0].month)
        for dt in dates[1:]:
            cur = (dt.year, dt.month)
            if cur != prev:
                rebal.append(dt)
            prev = cur

    elif freq == "Quarterly":
        prev = (dates[0].year, (dates[0].month - 1) // 3)
        for dt in dates[1:]:
            cur = (dt.year, (dt.month - 1) // 3)
            if cur != prev:
                rebal.append(dt)
            prev = cur

    else:
        raise ValueError(f"Unknown frequency '{freq}'. Use: Daily, Weekly, Monthly, Quarterly.")

    return rebal


def is_wash_sale_clear(symbol: str, check_date: pd.Timestamp,
                       wash_sale_log: dict, wash_sale_days: int) -> bool:
    """
    Return True if it is safe to BUY this symbol on check_date.
    A restriction exists if the symbol was sold within wash_sale_days of check_date.
    The wash_sale_log is shared across all positions in the portfolio.
    """
    if symbol not in wash_sale_log:
        return True
    return (check_date - wash_sale_log[symbol]).days > wash_sale_days


def log_event(
    position_id,
    rebalance_date,
    current_security,
    unrealized_return,
    position_value,
    event_occurred,
    reason,
    security_sold=None,
    security_bought=None,
    wash_sale_blocked=False,
) -> dict:
    """Build a single event log entry as a dict."""
    return {
        "position_id"      : position_id,
        "rebalance_date"   : rebalance_date,
        "current_security" : current_security,
        "unrealized_return": round(unrealized_return, 4),
        "position_value"   : round(position_value, 2),
        "event_occurred"   : event_occurred,
        "reason"           : reason,
        "security_sold"    : security_sold,
        "security_bought"  : security_bought,
        "wash_sale_blocked": wash_sale_blocked,
    }


print("Helper functions defined.")

## 6. Main Simulation Function

The simulation manages N positions simultaneously. On each rebalance date, it loops over every position and runs the same 5-case decision tree as the single-security version.

**Key difference from single-security:** the wash sale log is shared. If SPY is sold from the SPY position, no other position can buy SPY either until the window clears.

**Decision tree (applied independently to each position at each rebalance date):**
```
Check unrealized return on this position's current holding
│
├─ Loss < threshold → HOLD, log reason
│
└─ Loss ≥ threshold → Try to harvest
   │
   ├─ No proxy defined → HOLD, log reason
   ├─ Proxy price unavailable → HOLD, log reason
   ├─ Proxy in wash sale window → BLOCKED, log reason
   └─ All clear → SELL current, BUY proxy, log trade
```

In [ ]:
def run_multi_security_tlh(
    prices_wide: pd.DataFrame,
    proxy_df: pd.DataFrame,
    portfolio: dict,
    start_date: str,
    end_date: str,
    rebalance_freq: str,
    tlh_threshold: float,
    wash_sale_days: int,
    initial_value: float,
):
    """
    Simulate a multi-security TLH strategy.

    Parameters
    ----------
    prices_wide     : Wide DataFrame (dates x tickers) of daily closing prices
    proxy_df        : Filtered proxy lookup (order=1 rows only)
    portfolio       : Dict of {starting_symbol: normalized_weight}
    start_date      : Simulation start date string
    end_date        : Simulation end date string
    rebalance_freq  : 'Daily' | 'Weekly' | 'Monthly' | 'Quarterly'
    tlh_threshold   : Fractional loss that triggers a harvest (e.g. 0.03 = 3%)
    wash_sale_days  : Days a sold symbol is restricted from repurchase (portfolio-wide)
    initial_value   : Total starting portfolio value in dollars

    Returns
    -------
    event_log_df : DataFrame — one row per (position, rebalance_date), fully logged
    history_df   : DataFrame — one row per (position, trading_day) with daily values
    """

    # ── Validate inputs and prepare trading dates ─────────────────────────────
    start_ts = pd.Timestamp(start_date)
    end_ts   = pd.Timestamp(end_date)
    trading_dates = prices_wide.index[
        (prices_wide.index >= start_ts) & (prices_wide.index <= end_ts)
    ]

    if len(trading_dates) == 0:
        raise ValueError(f"No trading dates between {start_date} and {end_date}.")

    missing = [sym for sym in portfolio if sym not in prices_wide.columns]
    if missing:
        raise ValueError(f"These portfolio symbols are missing from price data: {missing}")

    # ── Initialize one position per portfolio symbol ──────────────────────────
    # Each position is a dict with: symbol (current holding), shares, cost_basis
    # position_id is the original starting symbol — it never changes even after a swap
    first_date = trading_dates[0]

    positions = {}
    for sym, weight in portfolio.items():
        pos_value    = initial_value * weight
        entry_price  = prices_wide.loc[first_date, sym]
        positions[sym] = {
            "symbol"    : sym,          # current holding (changes on TLH swap)
            "shares"    : pos_value / entry_price,
            "cost_basis": pos_value,    # resets to proceeds on every swap
        }

    # Shared wash sale log: {symbol: date_sold}
    # Applies portfolio-wide — selling a symbol in one position blocks it in all positions
    wash_sale_log = {}

    rebalance_dates = set(generate_rebalance_dates(trading_dates, rebalance_freq))

    event_log = []
    history   = []

    # ── Main loop — one pass per trading day ──────────────────────────────────
    for dt in trading_dates:

        # Daily snapshot: record every position's value
        for pos_id, pos in positions.items():
            cur_sym = pos["symbol"]
            price   = prices_wide.loc[dt, cur_sym]
            pos_val = pos["shares"] * price
            history.append({
                "date"          : dt,
                "position_id"   : pos_id,
                "current_symbol": cur_sym,
                "position_value": round(pos_val, 2),
            })

        # Only run TLH checks on rebalance dates
        if dt not in rebalance_dates:
            continue

        # ── Check each position independently ────────────────────────────────
        for pos_id, pos in positions.items():
            cur_sym = pos["symbol"]
            price   = prices_wide.loc[dt, cur_sym]
            pos_val = pos["shares"] * price

            unrealized_return = (pos_val - pos["cost_basis"]) / pos["cost_basis"]

            # Case 1: No trigger — hold
            if unrealized_return >= -tlh_threshold:
                event_log.append(log_event(
                    pos_id, dt, cur_sym, unrealized_return, pos_val,
                    event_occurred=False,
                    reason=(
                        f"No trigger — {unrealized_return:+.2%} "
                        f"above -{tlh_threshold:.0%} threshold"
                    ),
                ))
                continue

            # Loss breached threshold — try to harvest
            proxy = get_proxy(cur_sym, proxy_df)

            # Case 2: No proxy defined
            if proxy is None:
                event_log.append(log_event(
                    pos_id, dt, cur_sym, unrealized_return, pos_val,
                    event_occurred=False,
                    reason=(
                        f"TLH triggered ({unrealized_return:+.2%}) — "
                        f"no proxy defined for {cur_sym}, holding"
                    ),
                ))
                continue

            # Case 3: Proxy price not available
            if proxy not in prices_wide.columns or pd.isna(prices_wide.loc[dt, proxy]):
                event_log.append(log_event(
                    pos_id, dt, cur_sym, unrealized_return, pos_val,
                    event_occurred=False,
                    reason=(
                        f"TLH triggered ({unrealized_return:+.2%}) — "
                        f"{proxy} price unavailable on {dt.date()}, holding"
                    ),
                ))
                continue

            # Case 4: Proxy is in the wash sale window (portfolio-wide check)
            if not is_wash_sale_clear(proxy, dt, wash_sale_log, wash_sale_days):
                sold_on   = wash_sale_log[proxy].date()
                days_left = wash_sale_days - (dt - wash_sale_log[proxy]).days
                event_log.append(log_event(
                    pos_id, dt, cur_sym, unrealized_return, pos_val,
                    event_occurred=False,
                    reason=(
                        f"TLH triggered ({unrealized_return:+.2%}) — BLOCKED: "
                        f"{proxy} in wash sale window (sold {sold_on}, "
                        f"{days_left} days remaining)"
                    ),
                    wash_sale_blocked=True,
                ))
                continue

            # Case 5: Execute the TLH swap
            sold_sym    = cur_sym
            proxy_price = prices_wide.loc[dt, proxy]
            proceeds    = pos_val

            # Mark the sold symbol as restricted (affects ALL positions)
            wash_sale_log[sold_sym] = dt

            # Update this position to hold the proxy
            pos["symbol"]     = proxy
            pos["shares"]     = proceeds / proxy_price
            pos["cost_basis"] = proceeds   # new lot — basis resets to proceeds

            # Update today's history entry for this position
            for h in reversed(history):
                if h["date"] == dt and h["position_id"] == pos_id:
                    h["current_symbol"] = proxy
                    h["position_value"] = round(proceeds, 2)
                    break

            event_log.append(log_event(
                pos_id, dt, sold_sym, unrealized_return, proceeds,
                event_occurred=True,
                reason=(
                    f"TLH executed — loss {unrealized_return:+.2%} "
                    f"exceeded -{tlh_threshold:.0%} threshold"
                ),
                security_sold=sold_sym,
                security_bought=proxy,
            ))

    # ── Assemble output DataFrames ────────────────────────────────────────────
    event_log_df = pd.DataFrame(event_log)
    history_df   = pd.DataFrame(history)

    return event_log_df, history_df


print("Simulation function defined.")

## 7. Run the Prototype

In [ ]:
print("Running multi-security TLH prototype...")
print("-" * 55)

event_log, history = run_multi_security_tlh(
    prices_wide      = prices_wide,
    proxy_df         = proxy_df,
    portfolio        = PORTFOLIO,
    start_date       = START_DATE,
    end_date         = END_DATE,
    rebalance_freq   = REBALANCE_FREQ,
    tlh_threshold    = TLH_THRESHOLD,
    wash_sale_days   = WASH_SALE_DAYS,
    initial_value    = INITIAL_VALUE,
)

# Compute total portfolio value per day
totals = history.groupby("date")["position_value"].sum().reset_index()
totals.columns = ["date", "total_portfolio_value"]

print()
print("─── Portfolio-level results ───────────────────────")
final_value  = totals["total_portfolio_value"].iloc[-1]
total_return = final_value / INITIAL_VALUE - 1
print(f"  Initial value     : ${INITIAL_VALUE:>12,.0f}")
print(f"  Final value       : ${final_value:>12,.2f}")
print(f"  Total return      : {total_return:>+12.2%}")
print()
print("─── TLH activity summary ──────────────────────────")
n_checks  = len(event_log)
n_events  = int(event_log["event_occurred"].sum())
n_blocked = int(event_log["wash_sale_blocked"].sum())
print(f"  Rebalance checks  : {n_checks:>6}  ({n_checks // len(PORTFOLIO)} per position on average)")
print(f"  TLH events fired  : {n_events:>6}")
print(f"  Wash sale blocks  : {n_blocked:>6}")
print()
print("─── TLH events by position ────────────────────────")
per_pos = (
    event_log.groupby("position_id")["event_occurred"]
    .sum()
    .astype(int)
    .reset_index()
    .rename(columns={"event_occurred": "tlh_events"})
)
ws_per_pos = (
    event_log.groupby("position_id")["wash_sale_blocked"]
    .sum()
    .astype(int)
    .reset_index()
    .rename(columns={"wash_sale_blocked": "ws_blocks"})
)
summary = per_pos.merge(ws_per_pos, on="position_id")
for _, row in summary.iterrows():
    print(f"  {row['position_id']:<8}  TLH events: {row['tlh_events']:>2}   WS blocks: {row['ws_blocks']:>2}")

## 8. Event Log

Every rebalance check for every position is logged. Cells below show the full log, TLH events only, and wash sale blocks only.

In [ ]:
# Full event log (all positions, all rebalance dates)
display_log = event_log.copy()
display_log["rebalance_date"]    = display_log["rebalance_date"].dt.strftime("%Y-%m-%d")
display_log["unrealized_return"] = display_log["unrealized_return"].map("{:+.2%}".format)
display_log["position_value"]    = display_log["position_value"].map("${:,.0f}".format)
display_log["event_occurred"]    = display_log["event_occurred"].map({True: "YES", False: "no"})
display_log["wash_sale_blocked"] = display_log["wash_sale_blocked"].map({True: "BLOCKED", False: ""})

print(f"Full event log: {len(event_log)} rows ({len(PORTFOLIO)} positions × {len(event_log) // len(PORTFOLIO)} rebalance dates)")
display_log

In [ ]:
# TLH events only — the trades that actually happened
tlh_events = event_log[event_log["event_occurred"] == True].copy()

if tlh_events.empty:
    print("No TLH events occurred. Try lowering TLH_THRESHOLD.")
else:
    print(f"TLH events executed: {len(tlh_events)}")
    print()
    show = tlh_events[[
        "rebalance_date", "position_id", "security_sold",
        "security_bought", "unrealized_return", "position_value"
    ]].copy()
    show["rebalance_date"]    = show["rebalance_date"].dt.strftime("%Y-%m-%d")
    show["unrealized_return"] = show["unrealized_return"].map("{:+.2%}".format)
    show["position_value"]    = show["position_value"].map("${:,.0f}".format)
    show = show.reset_index(drop=True)
    show

In [ ]:
# Wash sale blocks only
blocked = event_log[event_log["wash_sale_blocked"] == True].copy()

if blocked.empty:
    print("No wash sale blocks occurred.")
else:
    print(f"Wash sale blocks: {len(blocked)}")
    print()
    show = blocked[["rebalance_date", "position_id", "current_security",
                    "unrealized_return", "reason"]].copy()
    show["rebalance_date"]    = show["rebalance_date"].dt.strftime("%Y-%m-%d")
    show["unrealized_return"] = show["unrealized_return"].map("{:+.2%}".format)
    show = show.reset_index(drop=True)
    show

## 9. Portfolio History

Daily portfolio value and per-position breakdowns across the simulation period.

In [ ]:
# ── Plot 1: Total portfolio NAV ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor("#0e1117")
fig.patch.set_facecolor("#0e1117")

ax.plot(totals["date"], totals["total_portfolio_value"],
        color="#1a73e8", linewidth=2)

# Mark TLH events on the chart
if not tlh_events.empty:
    event_totals = totals.set_index("date")["total_portfolio_value"]
    event_dates_unique = tlh_events["rebalance_date"].unique()
    for ed in event_dates_unique:
        ax.axvline(ed, color="yellow", alpha=0.25, linewidth=1, linestyle="--")
    ev_vals = [event_totals.loc[ed] if ed in event_totals.index else np.nan
               for ed in event_dates_unique]
    ax.scatter(event_dates_unique, ev_vals,
               color="yellow", zorder=5, s=50, label="TLH event (any position)")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.2f}M"))
ax.set_title(
    f"Total Portfolio NAV — {len(PORTFOLIO)}-security TLH  |  "
    f"{REBALANCE_FREQ} rebalance  |  {TLH_THRESHOLD:.0%} threshold",
    color="white", fontsize=12, fontweight="bold", pad=10,
)
ax.set_ylabel("Portfolio Value", color="#cccccc")
ax.tick_params(colors="#cccccc")
for spine in ["top", "right"]: ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]: ax.spines[spine].set_color("#333333")
ax.grid(True, alpha=0.12, color="#444444")
if not tlh_events.empty:
    ax.legend(facecolor="#1a1d23", labelcolor="white", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Per-position NAV grid ─────────────────────────────────────────────
n_pos   = len(PORTFOLIO)
n_cols  = 2
n_rows  = (n_pos + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
fig.patch.set_facecolor("#0e1117")
axes_flat = axes.flatten()

palette = ["#1a73e8", "#e8710a", "#34a853", "#ea4335",
           "#9c27b0", "#00bcd4", "#ff9800", "#8bc34a", "#f06292", "#80cbc4"]

for idx, (pos_id, color) in enumerate(zip(PORTFOLIO.keys(), palette)):
    ax = axes_flat[idx]
    ax.set_facecolor("#0e1117")

    pos_hist = history[history["position_id"] == pos_id].copy()
    ax.plot(pos_hist["date"], pos_hist["position_value"],
            color=color, linewidth=1.5)

    # Mark this position's TLH events
    pos_events = event_log[
        (event_log["position_id"] == pos_id) & (event_log["event_occurred"])
    ]
    if not pos_events.empty:
        # Get position value on event dates
        ev_vals = pos_hist.set_index("date")["position_value"]
        for _, ev in pos_events.iterrows():
            ax.axvline(ev["rebalance_date"], color="yellow", alpha=0.3,
                       linewidth=1, linestyle="--")
        ev_dates = pos_events["rebalance_date"].values
        ev_pv    = [ev_vals.loc[d] if d in ev_vals.index else np.nan for d in ev_dates]
        ax.scatter(ev_dates, ev_pv, color="yellow", s=40, zorder=5)

    # Show current holding (may have changed due to TLH)
    last_sym = pos_hist["current_symbol"].iloc[-1]
    start_sym = pos_id
    title_suffix = f" → {last_sym}" if last_sym != start_sym else ""
    n_swaps = int(event_log[event_log["position_id"] == pos_id]["event_occurred"].sum())

    ax.set_title(
        f"{pos_id}{title_suffix}  ({n_swaps} swap{'s' if n_swaps != 1 else ''})",
        color="white", fontsize=10, fontweight="bold",
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
    ax.tick_params(colors="#cccccc", labelsize=8)
    for spine in ["top", "right"]: ax.spines[spine].set_visible(False)
    for spine in ["bottom", "left"]: ax.spines[spine].set_color("#333333")
    ax.grid(True, alpha=0.10, color="#444444")

# Hide any unused subplots
for idx in range(n_pos, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle("Per-Position NAV  (yellow = TLH swap executed)",
             color="white", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Stacked area — which symbol is held per position over time ────────
# Shows switching behavior across all positions simultaneously
fig, axes = plt.subplots(n_pos, 1, figsize=(16, n_pos * 0.9), sharex=True)
fig.patch.set_facecolor("#0e1117")

for idx, (pos_id, color) in enumerate(zip(PORTFOLIO.keys(), palette)):
    ax = axes[idx]
    ax.set_facecolor("#0e1117")

    pos_hist = history[history["position_id"] == pos_id].copy()

    # Map symbol names to y-values for scatter
    unique_syms = pos_hist["current_symbol"].unique().tolist()
    sym_y       = {s: i for i, s in enumerate(unique_syms)}
    y_vals      = pos_hist["current_symbol"].map(sym_y)

    ax.scatter(pos_hist["date"], y_vals, c=color, s=3, alpha=0.8)
    ax.set_yticks(list(sym_y.values()))
    ax.set_yticklabels(list(sym_y.keys()), color="#cccccc", fontsize=8)
    ax.set_ylabel(pos_id, color="#aaaaaa", fontsize=9, rotation=0,
                  labelpad=40, va="center")
    ax.tick_params(colors="#cccccc", left=False, bottom=False)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(False)

axes[-1].tick_params(bottom=True, colors="#cccccc", labelsize=8)
plt.suptitle("Security Held Over Time (each row = one position)",
             color="white", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
final_vals      = history.groupby("date")["position_value"].sum()
total_return    = final_vals.iloc[-1] / INITIAL_VALUE - 1
n_days          = len(final_vals)
years           = n_days / 252.0
cagr            = (final_vals.iloc[-1] / INITIAL_VALUE) ** (1 / years) - 1
daily_rets      = final_vals.pct_change().dropna()
ann_vol         = daily_rets.std() * (252 ** 0.5)
running_max     = final_vals.cummax()
drawdown        = (final_vals - running_max) / running_max
max_drawdown    = drawdown.min()

print("Portfolio Summary")
print("-" * 40)
print(f"  Initial value     : ${INITIAL_VALUE:>14,.0f}")
print(f"  Final value       : ${final_vals.iloc[-1]:>14,.2f}")
print(f"  Total return      : {total_return:>+14.2%}")
print(f"  CAGR              : {cagr:>+14.2%}")
print(f"  Annualized vol    : {ann_vol:>14.2%}")
print(f"  Max drawdown      : {max_drawdown:>14.2%}")
print(f"  Sharpe (approx)   : {cagr/ann_vol:>14.2f}" if ann_vol > 0 else "  Sharpe           :            N/A")
print()
print(f"  TLH events fired  : {n_events:>14}")
print(f"  Wash sale blocks  : {n_blocked:>14}")
print(f"  Rebalance checks  : {n_checks:>14}")
print()

# Per-position final values
print("Per-position final values:")
pos_finals = (
    history[history["date"] == history["date"].max()]
    .set_index("position_id")
)
print(f"  {'Position':<10} {'Current Holding':<18} {'Final Value':>14} {'Return':>9}")
print(f"  {'─'*55}")
for pos_id, weight in PORTFOLIO.items():
    row      = pos_finals.loc[pos_id]
    fv       = row["position_value"]
    start_v  = INITIAL_VALUE * weight
    ret      = fv / start_v - 1
    print(f"  {pos_id:<10} {row['current_symbol']:<18} ${fv:>12,.0f} {ret:>+9.2%}")

## 10. Notes for Future Extension

---

### Near-term extensions

**Weight rebalancing between positions**  
Currently each position is independent — no rebalancing of weights occurs between them. A natural next step is adding a portfolio-level rebalance step: after TLH checks, bring weights back toward targets (using the existing `build_rebalanced_series` engine from the main backtest notebook).

**Secondary proxies (order > 1)**  
When the primary proxy is wash-sale blocked, try `order=2`, then `order=3`. Modify `get_proxy` to return a ranked list and attempt each in order.

**Lot-level accounting**  
Currently each position is a single lot with one cost basis. A real system tracks individual lots (entry date, entry price, shares) and selects which lots to harvest — e.g. highest-loss-first (specific identification).

**Short-term vs. long-term gain distinction**  
Track `entry_date` per lot. Distinguish ST (<1 year) from LT (≥1 year) losses. Different tax rates apply, and ST harvesting is generally more valuable.

**Tax savings quantification**  
On each TLH event, record `harvested_loss = cost_basis - proceeds`. Multiply by the applicable tax rate to get `tax_saved`. Accumulate across events for a total tax benefit figure.

**Dividends**  
The repo has `dividend_data.csv`. Reinvesting dividends updates `shares` on ex-dividend dates and changes the cost basis of existing lots.

**Transaction costs**  
Add a `transaction_cost_bps` parameter. On each TLH swap, reduce proceeds by `cost_bps / 10_000 * proceeds` before reinvesting.

---

### Structure to preserve

- The per-position loop at each rebalance date is the right decomposition — keep it explicit
- The shared `wash_sale_log` is the key IRS-correctness invariant — don't split it per position
- `log_event` is the single source of truth for the event log schema — extend it there, not ad hoc
- The `position_id` = original starting symbol is a useful human-readable key — keep it even as holdings change